# * Re-Org 2025 : Revenue & Subscriber
    (GX2, GX3, GX5, GX6)

In [1]:
import configparser
import datetime as dt
import pandas as pd
import numpy as np
import oracledb
import re

config = configparser.ConfigParser()
config.read('../../my_config.ini')
config.sections()

TDMDBPR_user = config['TDMDBPR']['username']
TDMDBPR_pwd = config['TDMDBPR']['password']
TDMDBPR_db = config['TDMDBPR']['db']
TDMDBPR_host = config['TDMDBPR']['host']
TDMDBPR_port = config['TDMDBPR']['port']

AKPIPRD_user = config['AKPIPRD']['username']
AKPIPRD_pwd = config['AKPIPRD']['password']
AKPIPRD_db = config['AKPIPRD']['db']
AKPIPRD_host = config['AKPIPRD']['host']
AKPIPRD_port = config['AKPIPRD']['port']

curr_dt = dt.datetime.now().date()
str_curr_dt = curr_dt.strftime('%Y%m%d')
curr_dt

datetime.date(2026, 5, 19)

## ETL Process...

### Step 1 : Import Data Source

In [2]:
''' Execute transaction '''


curr_datetime = dt.datetime.now().strftime('%Y-%m-%d, %H:%M:%S')
print(f'\nData as of {curr_datetime}')


# Connect : TDMDBPR
src_dsn = f'{TDMDBPR_user}/{TDMDBPR_pwd}@{TDMDBPR_host}:{TDMDBPR_port}/{TDMDBPR_db}'
src_conn = oracledb.connect(src_dsn)
src_cur = src_conn.cursor()

query = (f"""
         
    WITH W_PARAM AS
    (
        SELECT 20250101 AS V_INI_DAY_START
            , 20251231 AS V_INI_DAY_END
        FROM DUAL
    ) -->> W_PARAM
    -----------------------------------------------------------------------------------------------------------------------

    , W_ORG AS 
    ( 
        SELECT DISTINCT ZONE_TYPE
            , ORGID_G, TDS_SGMD
            , ORGID_H, HOP_HINT
        FROM CDSAPPO.DIM_MOOC_AREA
        WHERE TEAM_CODE <> 'ไม่ระบุ' AND REMARK <> 'Dummy'
        AND ORGID_G IN ('GX2' ,'GX3' ,'GX5', 'GX6') --Re-Org
    ) -->> W_ORG
    -----------------------------------------------------------------------------------------------------------------------

    , W_TXN_TEMP AS 
    (
        SELECT TM_KEY_MTH, TM_KEY_DAY, PRODUCT_GRP, METRIC_CD, METRIC_NAME, COMP_CD
            , 'G' AS AREA_TYPE, O.ORGID_G AS AREA_CD, O.TDS_SGMD AS AREA_NAME
            , SUM(ACTUAL_SNAP) AS ACTUAL_SNAP, SUM(TARGET_SNAP) AS TARGET_SNAP
            
        FROM GEOSPCAPPO.AGG_PERF_NEWCO A
        
        INNER JOIN W_ORG O
            ON O.ORGID_H = A.AREA_CD
            
        WHERE METRIC_CD IN (
         
            -->> Revenue
            'B2R010100' --Postpaid Revenue B2C
            , 'DB2R010100' --Postpaid Revenue B2C : DTAC
            , 'TB2R010100' --Postpaid Revenue B2C : TMH
            , 'B2R020100' --Postpaid Revenue B2B
            , 'DB2R020100' --Postpaid Revenue B2B : DTAC
            , 'TB2R020100' --Postpaid Revenue B2B : TMH
            , 'B1R000100' --Prepaid Revenue
            , 'DB1R000100' --Prepaid Revenue : DTAC
            , 'TB1R000100' --Prepaid Revenue : TMH
            , 'TB3R000100' --TOL Revenue
            , 'TB4R000100' --TVS Revenue
            
            -->> Subscriber
            , 'B2S010600' --Postpaid Reported SubBase B2C
            , 'DB2S010600' --Postpaid Reported SubBase B2C : DTAC
            , 'TB2S010600' --Postpaid Reported SubBase B2C : TMH
            , 'B1S000600' --Prepaid Active Caller 30D Rolling
            , 'B1S000601' --Prepaid Active Caller 30D Rolling : Thai Mass
            , 'B1S000602' --Prepaid Active Caller 30D Rolling : AEC/Migrants
            , 'B1S000603' --Prepaid Active Caller 30D Rolling : Tourists
            , 'DB1S000600' --Prepaid Active Caller 30D Rolling : DTAC
            , 'DB1S000601' --Prepaid Active Caller 30D Rolling : Thai Mass : DTAC
            , 'DB1S000602' --Prepaid Active Caller 30D Rolling : AEC/Migrants : DTAC
            , 'DB1S000603' --Prepaid Active Caller 30D Rolling : Tourists : DTAC
            , 'TB1S000600' --Prepaid Active Caller 30D Rolling : TMH
            , 'TB1S000601' --Prepaid Active Caller 30D Rolling : Thai Mass : TMH
            , 'TB1S000602' --Prepaid Active Caller 30D Rolling : AEC/Migrants : TMH
            , 'TB1S000603' --Prepaid Active Caller 30D Rolling : Tourists : TMH
            , 'TB3S000600' --FTTx Reported SubBase
            , 'TB4S000500' --TVS Active Subs
            )
        AND AREA_TYPE = 'H'
        AND TM_KEY_DAY BETWEEN (SELECT V_INI_DAY_START FROM W_PARAM) AND (SELECT V_INI_DAY_END FROM W_PARAM)
        
        GROUP BY TM_KEY_MTH, TM_KEY_DAY, PRODUCT_GRP, METRIC_CD, METRIC_NAME, COMP_CD, O.ORGID_G, O.TDS_SGMD
    ) -->> W_TXN_TEMP
    -----------------------------------------------------------------------------------------------------------------------

    , W_TXN_FIXED AS 
    (
        -->> Actual
        SELECT TM_KEY_MTH, TM_KEY_DAY, METRIC_CD, METRIC_NAME
            , ACTUAL_SNAP AS METRIC_VALUE
            , COMP_CD
            , 'A' AS VERSION, 2 AS AREA_NO
            , AREA_CD, AREA_NAME, AREA_TYPE
        FROM W_TXN_TEMP 
        
        UNION ALL
        
        -->> Target
        SELECT TM_KEY_MTH, TM_KEY_DAY, METRIC_CD, METRIC_NAME
            , TARGET_SNAP AS METRIC_VALUE
            , COMP_CD
            , 'T' AS VERSION, 2 AS AREA_NO
            , AREA_CD, AREA_NAME, AREA_TYPE
        FROM W_TXN_TEMP 
    ) -->> W_TXN_FIXED
    -----------------------------------------------------------------------------------------------------------------------
    -----------------------------------------------------------------------------------------------------------------------

         
    SELECT /*+PARALLEL(8)*/ 
        NULL AS TM_KEY_YR
        , TM_KEY_MTH
        , NULL TRUE_TM_KEY_WK
        , TM_KEY_DAY, METRIC_CD, METRIC_NAME, COMP_CD, VERSION, AREA_NO, AREA_TYPE, AREA_CD, AREA_NAME
        , METRIC_VALUE AS DAY_VALUE
        , NULL AS MTH_VALUE, NULL AS AGG_TYPE, NULL AS FREQUENCY
        , 'Re-Org 2025 to align with the 2026' AS REMARK 
    FROM W_TXN_FIXED 
    WHERE METRIC_VALUE IS NOT NULL
         
""")


try:
    # Create Dataframe
    src_cur.execute(query)
    rows = src_cur.fetchall()
    src_df = pd.DataFrame.from_records(rows, columns=[x[0] for x in src_cur.description])
    print(f'\nDataFrame: {src_df.shape[0]} rows, {src_df.shape[1]} columns')
    src_cur.close()


except oracledb.DatabaseError as e:
    print(f'\nError with Oracle : {e}')


finally:
    src_conn.close()


Data as of 2026-05-19, 14:51:20

DataFrame: 45108 rows, 17 columns


In [3]:
src_df

,TM_KEY_YR,TM_KEY_MTH,TRUE_TM_KEY_WK,TM_KEY_DAY,METRIC_CD,METRIC_NAME,COMP_CD,VERSION,AREA_NO,AREA_TYPE,AREA_CD,AREA_NAME,DAY_VALUE,MTH_VALUE,AGG_TYPE,FREQUENCY,REMARK
0,None,202506,None,20250618,DB2S010600,Postpaid Reported SubBase B2C : DTAC,DTAC,A,2,G,GX2,BMA-East,1.054683e+06,None,None,None,Re-Org 2025 to align with the 2026
1,None,202507,None,20250709,DB2S010600,Postpaid Reported SubBase B2C : DTAC,DTAC,A,2,G,GX2,BMA-East,1.060350e+06,None,None,None,Re-Org 2025 to align with the 2026
2,None,202509,None,20250908,DB2R010100,Postpaid Revenue B2C : DTAC,DTAC,A,2,G,GX6,Northeast2,6.418556e+06,None,None,None,Re-Org 2025 to align with the 2026
3,None,202508,None,20250802,DB2S010600,Postpaid Reported SubBase B2C : DTAC,DTAC,A,2,G,GX2,BMA-East,1.052788e+06,None,None,None,Re-Org 2025 to align with the 2026
4,None,202505,None,20250513,DB2R010100,Postpaid Revenue B2C : DTAC,DTAC,A,2,G,GX5,Northeast1,4.712289e+06,None,None,None,Re-Org 2025 to align with the 2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45103,None,202512,None,20251228,TB4R000100,TVS Revenue,TRUE,T,2,G,GX5,Northeast1,7.133110e+05,None,None,None,Re-Org 2025 to align with the 2026
45104,None,202501,None,20250128,DB2R020100,Postpaid Revenue B2B : DTAC,DTAC,T,2,G,GX3,East,1.765959e+04,None,None,None,Re-Org 2025 to align with the 2026
45105,None,202502,None,20250210,TB4R000100,TVS Revenue,TRUE,T,2,G,GX3,East,3.216545e+06,None,None,None,Re-Org 2025 to align with the 2026
45106,None,202507,None,20250722,TB2R010100,Postpaid Revenue B2C : TMH,TRUE,T,2,G,GX6,Northeast2,4.242941e+07,None,None,None,Re-Org 2025 to align with the 2026


### Insert to "INTERIM_VINSIGHT_DATA"
    Delete -> Insert

In [4]:
''' Input Parameter '''

# Create list
month_list = src_df['TM_KEY_MTH'].drop_duplicates().tolist()
version_list = src_df['VERSION'].drop_duplicates().tolist()
mt_cd_list = src_df['METRIC_CD'].drop_duplicates().tolist()

if len(version_list) == 1:
    version_list = str(version_list).replace(r'[', '(').replace(r']', ')')
else:
    version_list = tuple(version_list)

if len(mt_cd_list) == 1:
    mt_cd_list = str(mt_cd_list).replace(r'[', '(').replace(r']', ')')
else:
    mt_cd_list = tuple(mt_cd_list)

# Create Param
v_param = dict(mth_start=min(month_list), mth_end=max(month_list), version=version_list, metric_cd=mt_cd_list)
# v_param = dict(mth_start=202406, mth_end=202408, metric_cd=mt_cd_list)
v_target_schema = 'AUTOKPI'
v_target_table = 'INTERIM_VINSIGHT_DATA'

query_delete = f"""
    DELETE {v_target_schema}.{v_target_table} 
    WHERE VERSION IN {v_param['version']}
    AND METRIC_CD IN {v_param['metric_cd']}
    AND TM_KEY_MTH BETWEEN {v_param['mth_start']} AND {v_param['mth_end']} 
"""

print(f"\nParameter...\n\n   -> TM_KEY_MTH BETWEEN {v_param['mth_start']} AND {v_param['mth_end']}\n   -> VERSION IN {v_param['version']}\n   -> METRIC_CD IN {v_param['metric_cd']}")
print(f'\nDataFrame...\n\n   -> src_df : {src_df.shape[0]} rows, {src_df.shape[1]} columns') 
print(f'\nquery_delete...\n{query_delete}')


Parameter...

   -> TM_KEY_MTH BETWEEN 202501 AND 202512
   -> VERSION IN ('A', 'T')
   -> METRIC_CD IN ('DB2S010600', 'DB2R010100', 'DB1S000603', 'DB1S000600', 'TB1S000600', 'DB2R020100', 'B1S000601', 'TB2R010100', 'B1S000602', 'B2S010600', 'TB1S000602', 'DB1R000100', 'DB1S000602', 'B1S000600', 'TB1S000601', 'TB2S010600', 'TB1S000603', 'B2R010100', 'B1S000603', 'TB4S000500', 'B1R000100', 'TB1R000100', 'TB3R000100', 'TB3S000600', 'TB4R000100', 'DB1S000601')

DataFrame...

   -> src_df : 45108 rows, 17 columns

query_delete...

    DELETE AUTOKPI.INTERIM_VINSIGHT_DATA 
    WHERE VERSION IN ('A', 'T')
    AND METRIC_CD IN ('DB2S010600', 'DB2R010100', 'DB1S000603', 'DB1S000600', 'TB1S000600', 'DB2R020100', 'B1S000601', 'TB2R010100', 'B1S000602', 'B2S010600', 'TB1S000602', 'DB1R000100', 'DB1S000602', 'B1S000600', 'TB1S000601', 'TB2S010600', 'TB1S000603', 'B2R010100', 'B1S000603', 'TB4S000500', 'B1R000100', 'TB1R000100', 'TB3R000100', 'TB3S000600', 'TB4R000100', 'DB1S000601')
    AND TM_KE

In [5]:
''' DELETE -> INSERT '''

job_start_datetime = dt.datetime.now().strftime('%Y-%m-%d, %H:%M:%S')
print(f'\nJob Start... {job_start_datetime}')


# Create rows from DataFrame
rows = [tuple(x) for x in src_df.values]


# Connect : AKPIPRD
dsn = f'{AKPIPRD_user}/{AKPIPRD_pwd}@{AKPIPRD_host}:{AKPIPRD_port}/{AKPIPRD_db}'
conn = oracledb.connect(dsn)
print(f'\n{AKPIPRD_db} : Connected')
cur = conn.cursor()
print(f'\nProcessing...')


try:
    # # Truncate
    # cur.execute(f"TRUNCATE TABLE {v_target_schema}.{v_target_table}")
    # print(f'\n   -> TRUNCATE : "{v_target_table}" : Done !')

    # Delete
    cur.execute(query_delete)
    print(f'\n   -> DELETE : "{v_target_table}" : Done !')
    
    # Insert
    cur.executemany(f"""
        INSERT INTO {v_target_table} 
        (TM_KEY_YR, TM_KEY_MTH, TRUE_TM_KEY_WK, TM_KEY_DAY, METRIC_CD, METRIC_NAME, COMP_CD, VERSION, AREA_NO, AREA_TYPE, AREA_CD, AREA_NAME, DAY_VALUE, MTH_VALUE, AGG_TYPE, FREQUENCY, REMARK) 
        VALUES (:1,:2,:3,:4,:5,:6,:7,:8,:9,:10,:11,:12,:13,:14,:15,:16,:17)
        """, rows)
    print(f'\n   -> INSERT : "{v_target_table}" : Done !')

    cur.close()
    conn.commit()


except oracledb.DatabaseError as e:
    print(f'\nError with Oracle : {e}')


finally:
    conn.close()
    print(f'\n{AKPIPRD_db} : Disconnected')
    print(f'\nJob Done !!!')


Job Start... 2026-05-19, 14:57:26



AKPIPRD : Connected

Processing...

   -> DELETE : "INTERIM_VINSIGHT_DATA" : Done !

   -> INSERT : "INTERIM_VINSIGHT_DATA" : Done !

AKPIPRD : Disconnected

Job Done !!!
